In [1]:
import pandas as pd
from rdkit import Chem
def get_reactive_atom_indices(smiles):
    # 解析 SMILES
    mol = Chem.MolFromSmiles(smiles)
    
    # 获取标记为反应位点的原子
    reactive_atoms = []
    for atom in mol.GetAtoms():
        # 检查是否带有反应位点标记
        if atom.HasProp('molAtomMapNumber'):
            reactive_atoms.append(atom.GetIdx())  # 获取原子序号
    
    return reactive_atoms

In [2]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles_with_atom_map(smiles: str) -> str:
    try:
        # 将 SMILES 转换为分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("Invalid SMILES string.")
        
        # 提取位点信息（Atom Maps）
        atom_map = {atom.GetIdx(): atom.GetAtomMapNum() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}
        
        # 标准化分子（使用 Standardizer）
        uncharger = rdMolStandardize.Uncharger()  # 去质子化
        mol = uncharger.uncharge(mol)
        
        # 去除多余氢原子
        mol = Chem.RemoveHs(mol)
        
        # 恢复位点信息（Atom Maps）
        for idx, map_num in atom_map.items():
            mol.GetAtomWithIdx(idx).SetAtomMapNum(map_num)
        
        # 返回标准化后的 SMILES
        standardized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return standardized_smiles
    except Exception as e:
        return f"Error: {e}"

In [3]:
df = pd.read_csv('predict_site.csv')
substrates = df['SMILES'].to_list()
predicts = df['predict_site'].to_list()
predicts = [i.split('|') for i in predicts]
atomidxs = []
for i in predicts:
    atomidxs_part = []
    for j in i:
        try:
            j = standardize_smiles_with_atom_map(j)
            atomidx = get_reactive_atom_indices(j)
            atomidxs_part.append(atomidx)
        except:
            pass
    atomidxs.append(atomidxs_part)
df['atomidxs'] = atomidxs
df

[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Run

,SMILES,metabolite,predict_site,atomidxs
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@...,CC(=O)O[C@@]1(C#C)CC[C@@H]2[C@]1(CC)CC[C@@H]1[...,"[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)OC2O[C@H](...,CC([CH3:1])(O)C1=CC=CC=C1CC[C@@H](SCC1(CC(=O)O...,"[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2O)cc1|N...,CC(C)CNCC1=CC=C(C2=CC=CC=C2S(=O)(=O)N2CCC[CH2:...,"[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,CC(CN1c2ccccc2S(=O)c2ccccc21)N(C)C|CNC(C)CN1c2...,CC(CN1C2=CC=CC=C2[S:1]C2=CC=CC=C12)N(C)C|CC(CN...,"[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(C3=CC=C(C(=O)NS(=O)(=O)C4=CC=...,"[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,O=C(O)c1cc(O)c2c(c1)C(C1c3cccc(O)c3C(=O)c3c(O)...,C1=CC=C2C(=C1OC1C(O)C(O)C(O)C(CO)O1)C(=O)C1=C(...,"[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,O=C(O)[C@H]1OC(OC2CN=C(c3ccccc3)c3cc(Cl)ccc3N2...,O=C1[CH2:1]N=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C2N1|O...,"[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C(O)[C@H]1O[C@@H](OC2N=C(c3ccccc3)c3cc(Cl)cc...,O=C1NC2=CC=C(Cl)C=C2C(C2=CC=CC=C2)=NC1[OH:1]|C...,"[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,O=CCCl|NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1C...,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1(NCCCl)OCCCN1[...,"[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [4]:
df.to_pickle('ourmodel_results.pickle')

In [5]:
df = pd.read_pickle('ourmodel_results.pickle')
df

,SMILES,metabolite,predict_site,atomidxs
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@...,CC(=O)O[C@@]1(C#C)CC[C@@H]2[C@]1(CC)CC[C@@H]1[...,"[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)OC2O[C@H](...,CC([CH3:1])(O)C1=CC=CC=C1CC[C@@H](SCC1(CC(=O)O...,"[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2O)cc1|N...,CC(C)CNCC1=CC=C(C2=CC=CC=C2S(=O)(=O)N2CCC[CH2:...,"[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,CC(CN1c2ccccc2S(=O)c2ccccc21)N(C)C|CNC(C)CN1c2...,CC(CN1C2=CC=CC=C2[S:1]C2=CC=CC=C12)N(C)C|CC(CN...,"[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(C3=CC=C(C(=O)NS(=O)(=O)C4=CC=...,"[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,O=C(O)c1cc(O)c2c(c1)C(C1c3cccc(O)c3C(=O)c3c(O)...,C1=CC=C2C(=C1OC1C(O)C(O)C(O)C(CO)O1)C(=O)C1=C(...,"[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,O=C(O)[C@H]1OC(OC2CN=C(c3ccccc3)c3cc(Cl)ccc3N2...,O=C1[CH2:1]N=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C2N1|O...,"[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C(O)[C@H]1O[C@@H](OC2N=C(c3ccccc3)c3cc(Cl)cc...,O=C1NC2=CC=C(Cl)C=C2C(C2=CC=CC=C2)=NC1[OH:1]|C...,"[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,O=CCCl|NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1C...,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1(NCCCl)OCCCN1[...,"[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [6]:
subs = df['SMILES'].to_list()
atomidxs = df['atomidxs'].to_list()
dict_sub2atomidxs = dict(zip(subs,atomidxs))
dict_sub2atomidxs

{'C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC': [[17],
  [15],
  [6],
  [12],
  [4],
  [12],
  [4],
  [25],
  [26],
  [21],
  [4, 21],
  [12]],
 'CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc(/C=C/c2ccc3ccc(Cl)cc3n2)c1': [[40],
  [37, 38],
  [19],
  [14],
  [32],
  [40],
  [21],
  [12],
  [8],
  [40],
  [38, 39],
  [21],
  [7]],
 'CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1': [[23],
  [5],
  [1],
  [23],
  [5],
  [25],
  [22],
  [3],
  [4],
  [16],
  [4],
  [4],
  [4],
  [25],
  [25]],
 'CC(CN1c2ccccc2Sc2ccccc21)N(C)C': [[10],
  [10],
  [13],
  [10],
  [15],
  [14],
  [14, 15],
  [17],
  [17],
  [5],
  [13],
  [15, 16],
  [16],
  [13]],
 'CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NCC5CCOCC5)c([N+](=O)[O-])c4)c(Oc4cnc5[nH]ccc5c4)c3)CC2)=C(c2ccc(Cl)cc2)C1': [[26],
  [6],
  [34],
  [2],
  [15],
  [40],
  [60],
  [60],
  [31],
  [32],
  [59],
  [60],
  [18],
  [34],
  [50],
  [51],
  [7],
  [7],
  [45, 46]],
 'CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)

In [7]:
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_65_site_truth_correct_withatomidx_noenzyme.pickle')
substrates = df_total['substrates'].to_list()
ourmodel_results_old = [dict_sub2atomidxs[i] for i in substrates]
df_total['ourmodel_results_old'] = ourmodel_results_old
df_total

,substrates,true_sites,true_sites_old,gnn_results,smartcyp_results,xenosite_results,somp_results,metapredictor_results_old,ourmodel_results_old
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,[3],[3],"{0: 9, 1: 6, 2: 2, 3: 1, 4: 4, 5: 18, 6: 13, 7...","{0: 53, 1: 66, 2: 72, 3: 69, 4: 60, 5: 5, 6: 5...","{0: 65, 1: 107, 2: 138, 3: 77, 4: 107, 5: 74, ...","{0: 97, 1: 156, 2: 140, 3: 143, 4: 93, 5: 93, ...","[[3], [17], [15, 17], [12], [15], [13, 14], [1...","[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[0, 10, 19, 13]","[40, 32, 10, 19]","{0: 18, 1: 5, 2: 16, 3: 3, 4: 34, 5: 36, 6: 42...","{0: 36, 1: 83, 2: 36, 3: 75, 4: 89, 5: 44, 6: ...","{0: 68, 1: 204, 2: 75, 3: 36, 4: 184, 5: 112, ...","{0: 94, 1: 112, 2: 100, 3: 178, 4: 221, 5: 98,...","[[6], [19], [40], [7], [1, 40], [40], [1], [40...","[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,"[1, 2, 4, 5, 22, 23]","[1, 4, 5, 22, 23, 25]","{0: 15, 1: 22, 2: 16, 3: 7, 4: 2, 5: 6, 6: 23,...","{0: 26, 1: 21, 2: 26, 3: 6, 4: 18, 5: 13, 6: 5...","{0: 46, 1: 63, 2: 53, 3: 74, 4: 30, 5: 73, 6: ...","{0: 70, 1: 32, 2: 76, 3: 91, 4: 127, 5: 41, 6:...","[[23], [22], [25], [4], [1], [3], [3, 25], [13...","[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,"[17, 10, 14]","[10, 11, 12, 13, 14, 15, 16, 17]","{0: 12, 1: 6, 2: 4, 3: 5, 4: 16, 5: 13, 6: 11,...","{0: 26, 1: 11, 2: 18, 3: 30, 4: 39, 5: 24, 6: ...","{0: 56, 1: 58, 2: 64, 3: 22, 4: 108, 5: 65, 6:...","{0: 78, 1: 67, 2: 92, 3: 96, 4: 107, 5: 61, 6:...","[[13], [14], [10], [7], [11, 12, 13, 14, 15, 1...","[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,"[2, 7, 50, 59, 60]","[2, 7, 50, 59, 60]","{0: 34, 1: 42, 2: 32, 3: 45, 4: 29, 5: 20, 6: ...","{0: 68, 1: 108, 2: 68, 3: 65, 4: 42, 5: 126, 6...","{0: 120, 1: 289, 2: 127, 3: 125, 4: 149, 5: 21...","{0: 126, 1: 214, 2: 132, 3: 54, 4: 97, 5: 160,...","[[58], [2], [59], [45], [60], [25], [39], [34]...","[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...,...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,"[26, 47]","[30, 47]","{0: 48, 1: 7, 2: 28, 3: 9, 4: 14, 5: 1, 6: 2, ...","{0: 64, 1: 49, 2: 64, 3: 59, 4: 30, 5: 56, 6: ...","{0: 238, 1: 229, 2: 128, 3: 268, 4: 161, 5: 28...","{0: 208, 1: 162, 2: 276, 3: 272, 4: 115, 5: 20...","[[30], [11], [10, 30], [30, 47], [11, 30], [47...","[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,"[0, 1, 2]","[16, 17, 18]","{0: 3, 1: 1, 2: 5, 3: 2, 4: 6, 5: 18, 6: 9, 7:...","{0: 30, 1: 33, 2: 3, 3: 8, 4: 48, 5: 42, 6: 19...","{0: 50, 1: 84, 2: 50, 3: 19, 4: 74, 5: 92, 6: ...","{0: 52, 1: 85, 2: 13, 3: 54, 4: 101, 5: 71, 6:...","[[8], [16], [15], [18], [18], [9], [16, 17, 18...","[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,"[0, 1, 19]","[16, 17, 18, 19]","{0: 6, 1: 5, 2: 4, 3: 10, 4: 18, 5: 13, 6: 9, ...","{0: 30, 1: 33, 2: 27, 3: 42, 4: 20, 5: 11, 6: ...","{0: 55, 1: 90, 2: 20, 3: 103, 4: 60, 5: 39, 6:...","{0: 54, 1: 94, 2: 95, 3: 97, 4: 55, 5: 35, 6: ...","[[19], [14], [18], [15], [19], [18, 19], [5]]","[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,"[2, 9, 10, 11]","[2, 4, 9, 10, 12, 13]","{0: 6, 1: 8, 2: 1, 3: 2, 4: 11, 5: 12, 6: 13, ...","{0: 30, 1: 33, 2: 15, 3: 6, 4: 24, 6: 30, 7: 1...","{0: 54, 1: 39, 2: 23, 3: 46, 4: 41, 5: 70, 6: ...","{0: 49, 1: 75, 2: 55, 3: 24, 4: 61, 5: 39, 6: ...","[[2], [10], [12], [9], [4], [12, 13], [5], [2,...","[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [8]:
df_total.to_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_65_site_truth_correct_withatomidx_noenzyme.pickle')

In [3]:
import pandas as pd
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_123_site_truth_correct_withatomidx_noenzyme.pickle')

In [5]:
df_total

,substrate,predict,metabolite,site_truth,pred_site,sites,gnn_results,somp_results,xenosite_results,smartcyp_results
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,CC(/C=C/[C@]12O[C@]1(C)CCCC2(C)C)=C\C=C\C(C)=C...,CC(/C=C/C12C(C)(C)CCC[C:1]1(C)[O:1]2)=C\C=C\C(...,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,"[20, 22]","{0: 23, 1: 20, 2: 21, 3: 16, 4: 11, 5: 17, 6: ...","{0: 65, 1: 85, 2: 81, 3: 60, 4: 58, 5: 63, 6: ...","{0: 49, 1: 91, 2: 68, 3: 67, 4: 115, 5: 48, 6:...","{0: 14, 1: 60, 2: 32, 3: 6, 4: 57, 5: 49, 6: 4..."
1,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2|CC(=...,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NC(O)C(=O)N2,CC(=O)Nc1cc2c([cH:1]c1)NC(=O)CN=C2c1ccccc1Cl|C...,CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,[19],"{0: 13, 1: 3, 2: 17, 3: 6, 4: 14, 5: 11, 6: 20...","{0: 42, 1: 65, 2: 79, 3: 46, 4: 109, 5: 30, 6:...","{0: 42, 1: 63, 2: 57, 3: 15, 4: 95, 5: 38, 6: ...","{0: 7, 1: 42, 2: 39, 3: 29, 4: 54, 5: 19, 6: 3..."
2,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC1=C2C(=C(NC3=CC=C(I)C=C3F)N(C)C1=O)C(=O)N(C1...,Cc1c(=O)n(C)c(Nc2ccc(I)cc2F)c2c(=O)n(C3CC3)c(=...,CC(=O)N[c:1]1[cH:1][cH:1][cH:1][c:1](-n2c(=O)n...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,"[3, 2, 3, 36]","{0: 11, 1: 4, 2: 19, 3: 3, 4: 20, 5: 18, 6: 17...","{0: 74, 1: 145, 2: 171, 3: 89, 4: 215, 5: 47, ...","{0: 71, 1: 119, 2: 88, 3: 19, 4: 157, 5: 76, 6...","{0: 4, 1: 49, 2: 45, 3: 34, 4: 58, 5: 23, 6: 3..."
3,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)C1=CC(C(C)(C)CO)=C(O)C=C1NC(=O)C1=CNC2...,CC(C)(C)c1cc(C(C)(C)C(=O)O)c(O)cc1NC(=O)c1c[nH...,CC(C)(C)c1c(NC(=O)c2c[nH]c3ccccc3c2=O)cc([OH:1...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,[10],"{0: 8, 1: 14, 2: 16, 3: 11, 4: 6, 5: 15, 6: 19...","{0: 30, 1: 72, 2: 36, 3: 42, 4: 113, 5: 91, 6:...","{0: 47, 1: 145, 2: 54, 3: 60, 4: 133, 5: 87, 6...","{0: 16, 1: 39, 2: 16, 3: 16, 4: 45, 5: 29, 6: ..."
4,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CC(C)(CO)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=...,CC(C)(CO)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OC...,CC(C)(C)[c:1]1[cH:1][c:1](NC(=O)Nc2ccc(-c3cn4c...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],"{0: 12, 1: 8, 2: 11, 3: 7, 4: 28, 5: 29, 6: 40...","{0: 57, 1: 52, 2: 63, 3: 69, 4: 144, 5: 89, 6:...","{0: 12, 1: 40, 2: 13, 3: 14, 4: 37, 5: 36, 6: ...","{0: 22, 1: 62, 2: 22, 3: 22, 4: 70, 5: 26, 6: ..."
...,...,...,...,...,...,...,...,...,...,...
118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,OCC[N+]12CCC(C(O)(c3ccccc3)c3ccccc3)(CC1)CC2|O...,OC(c1ccccc1)(C12CC[N+](CCOCc3ccccc3)(CC1)CC2)[...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CC[O:1]Cc3cccc...,"[25, 20]","{0: 2, 1: 1, 2: 27, 3: 20, 4: 25, 5: 32, 6: 23...",None,"{0: 1, 1: 28, 2: 31, 3: 19, 4: 3, 5: 12, 6: 4,...","{0: 42, 1: 47, 2: 39, 3: 31, 4: 24, 5: 14, 6: ..."
119,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(O)C=C3)=...,O=C(O)[C@H]1OC(O[C@@H]2[C@@H](O)[C@H](c3ccc(Cl...,OC[C@H]1O[C@@H]([c:1]2ccc(Cl)c(Cc3ccc(OCCOC4CC...,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc([O:1]CCOC4CC...,"[28, 29, 19, 18, 19, 11]","{0: 2, 1: 6, 2: 26, 3: 32, 4: 21, 5: 17, 6: 12...","{0: 149, 1: 31, 2: 140, 3: 62, 4: 160, 5: 157,...","{0: 35, 1: 76, 2: 81, 3: 104, 4: 99, 5: 162, 6...","{0: 48, 1: 3, 2: 15, 3: 57, 4: 29, 5: 68, 6: 4..."
120,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,O=CC1=CN=C(CO)N1C1=CC=C(Cl)C=C1C(=O)C1=CC=CC=C...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2O|O=C(O...,Fc1ccccc1C1=NCc2cn[c:1](C[OH:1])n2-c2ccc(Cl)cc...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=N[CH2:1]2|...,"[15, 6, 23]","{0: 2, 1: 1, 2: 3, 3: 8, 4: 21, 5: 10, 6: 5, 7...","{0: 61, 1: 13, 2: 82, 3: 46, 4: 64, 5: 99, 6: ...","{0: 19, 1: 40, 2: 107, 3: 52, 4: 72, 5: 113, 6...","{0: 42, 1: 4, 2: 48, 3: 32, 4: 12, 5: 51, 6: 3..."
121,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,O=C(O)C1OC(Oc2ccc(COCC[N+]34

In [6]:
gnn_results = df_total['gnn_results'].to_list()
smartcyp_results = df_total['smartcyp_results'].to_list()
xenosite_results = df_total['xenosite_results'].to_list()
somp_results = df_total['somp_results'].to_list()

gnn_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in gnn_results]
smartcyp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in smartcyp_results]
xenosite_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in xenosite_results]
somp_sorted_atoms = []
for i in somp_results:
    try:
        sub_result = [k for k, v in sorted(i.items(), key=lambda item: item[1])]
    except:
        sub_result = ['None']
    somp_sorted_atoms.append(sub_result)
# somp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in somp_results]

df_total['gnn_results'] = gnn_sorted_atoms
df_total['smartcyp_results'] = smartcyp_sorted_atoms
df_total['xenosite_results'] = xenosite_sorted_atoms
df_total['somp_results'] = somp_sorted_atoms
df_total

,substrate,predict,metabolite,site_truth,pred_site,sites,gnn_results,somp_results,xenosite_results,smartcyp_results
0,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)O,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\C(=O)...,CC(/C=C/[C@]12O[C@]1(C)CCCC2(C)C)=C\C=C\C(C)=C...,CC(/C=C/C12C(C)(C)CCC[C:1]1(C)[O:1]2)=C\C=C\C(...,CC(/C=C/C12OC1(C)CCCC2(C)C)=C\C=C\C(C)=C\[C:1]...,"[20, 22]","[6, 7, 12, 19, 20, 9, 22, 13, 8, 11, 4, 10, 16...","[8, 9, 7, 10, 12, 13, 15, 4, 3, 5, 0, 6, 18, 2...","[19, 22, 14, 5, 16, 0, 15, 9, 8, 18, 12, 3, 2,...","[3, 9, 8, 18, 0, 10, 7, 19, 12, 13, 15, 2, 14,..."
1,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2,NC1=CC=C2C(=C1)C(C1=CC=CC=C1Cl)=NCC(=O)N2|CC(=...,CC(=O)Nc1ccc2c(c1)C(c1ccccc1Cl)=NC(O)C(=O)N2,CC(=O)Nc1cc2c([cH:1]c1)NC(=O)CN=C2c1ccccc1Cl|C...,CC(=O)[NH:1]c1ccc2c(c1)C(c1ccccc1Cl)=NCC(=O)N2...,[19],"[20, 18, 1, 22, 21, 3, 19, 10, 14, 17, 5, 9, 0...","[19, 14, 5, 13, 0, 3, 15, 6, 12, 21, 9, 1, 18,...","[3, 22, 18, 14, 5, 12, 0, 13, 15, 6, 9, 2, 21,...","[19, 0, 18, 14, 13, 5, 15, 12, 3, 9, 6, 22, 2,..."
2,CC(=O)Nc1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc4ccc(...,CC1=C2C(=C(NC3=CC=C(I)C=C3F)N(C)C1=O)C(=O)N(C1...,Cc1c(=O)n(C)c(Nc2ccc(I)cc2F)c2c(=O)n(C3CC3)c(=...,CC(=O)N[c:1]1[cH:1][cH:1][cH:1][c:1](-n2c(=O)n...,CC(=O)[NH:1]c1cccc(-n2c(=O)n(C3CC3)c(=O)c3c(Nc...,"[3, 2, 3, 36]","[29, 30, 3, 1, 25, 20, 9, 27, 7, 22, 0, 28, 31...","[30, 34, 5, 6, 23, 0, 22, 7, 24, 33, 26, 3, 29...","[3, 20, 23, 6, 22, 14, 0, 12, 15, 5, 29, 13, 3...","[0, 34, 23, 20, 26, 30, 7, 5, 14, 15, 22, 13, ..."
3,CC(C)(C)c1cc(C(C)(C)C)c(NC(=O)c2c[nH]c3ccccc3c...,CC(C)(C)C1=CC(C(C)(C)CO)=C(O)C=C1NC(=O)C1=CNC2...,CC(C)(C)c1cc(C(C)(C)C(=O)O)c(O)cc1NC(=O)c1c[nH...,CC(C)(C)c1c(NC(=O)c2c[nH]c3ccccc3c2=O)cc([OH:1...,CC(C)(C)c1cc(C(C)(C)[CH3:1])c(O)cc1NC(=O)c1c[n...,[10],"[13, 12, 14, 27, 28, 4, 15, 0, 26, 22, 3, 8, 9...","[0, 8, 2, 9, 21, 3, 10, 20, 26, 22, 7, 1, 19, ...","[12, 28, 17, 0, 20, 19, 2, 22, 3, 8, 21, 9, 25...","[28, 21, 20, 0, 2, 3, 17, 19, 22, 26, 8, 9, 10..."
4,CC(C)(C)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCC...,CC(C)(CO)C1=CC(NC(=O)NC2=CC=C(C3=CN4C(=N3)SC3=...,CC(C)(CO)c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OC...,CC(C)(C)[c:1]1[cH:1][c:1](NC(=O)Nc2ccc(-c3cn4c...,CC(C)(c1cc(NC(=O)Nc2ccc(-c3cn4c(n3)sc3cc(OCCN5...,[39],"[31, 29, 28, 32, 17, 8, 3, 1, 30, 27, 2, 0, 16...","[20, 1, 26, 0, 2, 25, 3, 29, 31, 12, 5, 37, 28...","[7, 30, 38, 10, 19, 27, 17, 9, 29, 31, 39, 0, ...","[28, 32, 26, 29, 31, 20, 27, 34, 0, 2, 3, 5, 1..."
...,...,...,...,...,...,...,...,...,...,...
118,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CCOCc3ccccc3)(...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,OCC[N+]12CCC(C(O)(c3ccccc3)c3ccccc3)(CC1)CC2|O...,OC(c1ccccc1)(C12CC[N+](CCOCc3ccccc3)(CC1)CC2)[...,OC(c1ccccc1)(c1ccccc1)C12CC[N+](CC[O:1]Cc3cccc...,"[25, 20]","[1, 0, 19, 21, 20, 17, 16, 18, 30, 28, 25, 9, ...",[None],"[0, 20, 4, 6, 10, 12, 30, 17, 31, 16, 28, 5, 1...","[18, 17, 21, 5, 11, 19, 25, 16, 28, 30, 24, 26..."
119,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc(OCCOC4CC4)cc...,OC[C@H]1O[C@@H](C2=CC=C(Cl)C(CC3=CC=C(O)C=C3)=...,O=C(O)[C@H]1OC(O[C@@H]2[C@@H](O)[C@H](c3ccc(Cl...,OC[C@H]1O[C@@H]([c:1]2ccc(Cl)c(Cc3ccc(OCCOC4CC...,OC[C@H]1O[C@@H](c2ccc(Cl)c(Cc3ccc([O:1]CCOC4CC...,"[28, 29, 19, 18, 19, 11]","[27, 0, 15, 26, 16, 1, 14, 28, 13, 17, 18, 6, ...","[19, 1, 23, 24, 20, 3, 9, 6, 13, 29, 32, 35, 2...","[0, 29, 31, 27, 20, 30, 1, 14, 2, 19, 11, 21, ...","[1, 20, 30, 2, 28, 18, 21, 22, 11, 26, 17, 4, ..."
120,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2,O=CC1=CN=C(CO)N1C1=CC=C(Cl)C=C1C(=O)C1=CC=CC=C...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=NC2O|O=C(O...,Fc1ccccc1C1=NCc2cn[c:1](C[OH:1])n2-c2ccc(Cl)cc...,OCc1ncc2n1-c1ccc(Cl)cc1C(c1ccccc1F)=N[CH2:1]2|...,"[15, 6, 23]","[1, 0, 2, 22, 6, 16, 18, 3, 17, 5, 23, 19, 14,...","[1, 23, 18, 9, 17, 3, 8, 16, 19, 0, 4, 12, 2, ...","[0, 22, 18, 17, 1, 9, 16, 19, 6, 3, 23, 12, 4,...","[1, 23, 18, 4, 22, 17, 9, 19, 16, 3, 12, 8, 6,..."
121,Oc1ccc(COCC[N+]23CCC(C(O)(c4ccccc4)c4ccccc4)(C...,OCC[N+]12CCC(C(O)(C3=CC=CC=C3)C3=CC=CC=C3)(CC1...,O=C(O)C1OC(Oc2ccc(COCC[N+]

In [7]:
df_total.to_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_123_site_truth_correct_withrank_noenzyme.pickle')

In [8]:
def get_unique_numbers(nums, n):
    """
    从列表中提取前 n 个不同的数字。

    参数:
        nums (list): 输入的数字列表。
        n (int): 需要提取的不同数字的数量。

    返回:
        list: 包含前 n 个不同数字的列表。
    """
    unique_nums = []
    seen = set()

    for num in nums:
        if num not in seen:
            unique_nums.append(num)
            seen.add(num)
        if len(unique_nums) == n:
            break

    return unique_nums

# 示例使用
nums = [1, 2, 2, 3, 4, 3, 5, 1, 6]
n = 4
result = get_unique_numbers(nums, n)
print(result)  # 输出: [1, 2, 3, 4]


[1, 2, 3, 4]


In [9]:
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_123_site_truth_correct_withrank_noenzyme.pickle')

# metapredictor_results_old = df_total['metapredictor_results_old'].to_list()
# ourmodel_results_old = df_total['ourmodel_results_old'].to_list()
# ourmodel_results_new = []
# for i in ourmodel_results_old:
#     list_part = []
#     for j in i:
#         list_part = list_part + j
#     ourmodel_results_new.append(list_part)
# metapredictor_results_new = []
# for i in metapredictor_results_old:
#     list_part = []
#     for j in i:
#         list_part = list_part + j
#     metapredictor_results_new.append(list_part)

# true_sites_old = df_total['true_sites_old'].to_list()
true_sites = df_total['sites'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

# def acc_top_n_old(n,modelresults,true_sites):
#     total_num = len(true_sites)
#     correct_num = 0
#     for idx,i in enumerate(true_sites):
#         modelresult = modelresults[idx]
#         modelresult = get_unique_numbers(modelresult, n)
#         modelresult = set(modelresult)
#         i = set(i)
#         intersection = i & modelresult
#         if len(intersection) > 0:
#             correct_num = correct_num + 1
#     acc = correct_num/total_num
#     print(f'top {n} accuracy = {acc}')
#     return acc

gnn_top1_acc = acc_top_n(1,gnn_sorted_atoms,true_sites)
gnn_top3_acc = acc_top_n(3,gnn_sorted_atoms,true_sites)
gnn_top5_acc = acc_top_n(5,gnn_sorted_atoms,true_sites)
gnn_topn = []
gnn_topn.append(gnn_top1_acc)
gnn_topn.append(gnn_top3_acc)
gnn_topn.append(gnn_top5_acc)

smartcyp_top1_acc = acc_top_n(1,smartcyp_sorted_atoms,true_sites)
smartcyp_top3_acc = acc_top_n(3,smartcyp_sorted_atoms,true_sites)
smartcyp_top5_acc = acc_top_n(5,smartcyp_sorted_atoms,true_sites)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

xenosite_top1_acc = acc_top_n(1,xenosite_sorted_atoms,true_sites)
xenosite_top3_acc = acc_top_n(3,xenosite_sorted_atoms,true_sites)
xenosite_top5_acc = acc_top_n(5,xenosite_sorted_atoms,true_sites)
xenosite_topn = []
xenosite_topn.append(xenosite_top1_acc)
xenosite_topn.append(xenosite_top3_acc)
xenosite_topn.append(xenosite_top5_acc)

somp_top1_acc = acc_top_n(1,somp_sorted_atoms,true_sites)
somp_top3_acc = acc_top_n(3,somp_sorted_atoms,true_sites)
somp_top5_acc = acc_top_n(5,somp_sorted_atoms,true_sites)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

# metapredictor_top1_acc = acc_top_n_old(1,metapredictor_results_new,true_sites_old)
# metapredictor_top3_acc = acc_top_n_old(3,metapredictor_results_new,true_sites_old)
# metapredictor_top5_acc = acc_top_n_old(5,metapredictor_results_new,true_sites_old)
# metapredictor_topn = []
# metapredictor_topn.append(metapredictor_top1_acc)
# metapredictor_topn.append(metapredictor_top3_acc)
# metapredictor_topn.append(metapredictor_top5_acc)

# ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
# ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
# ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
# ourmodel_topn = []
# ourmodel_topn.append(ourmodel_top1_acc)
# ourmodel_topn.append(ourmodel_top3_acc)
# ourmodel_topn.append(ourmodel_top5_acc)

top 1 accuracy = 0.13821138211382114
top 3 accuracy = 0.43089430894308944
top 5 accuracy = 0.6097560975609756
top 1 accuracy = 0.10569105691056911
top 3 accuracy = 0.3008130081300813
top 5 accuracy = 0.45528455284552843
top 1 accuracy = 0.2682926829268293
top 3 accuracy = 0.5040650406504065
top 5 accuracy = 0.6341463414634146
top 1 accuracy = 0.15447154471544716
top 3 accuracy = 0.35772357723577236
top 5 accuracy = 0.43089430894308944


In [10]:
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['gnnsom_topn'] = gnn_topn
df['smartcyp_topn'] = smartcyp_topn
df['xenosite_topn'] = xenosite_topn
df['somp_topn'] = somp_topn
# df['metapredictor_topn'] = metapredictor_topn
# df['ourmodel_topn'] = ourmodel_topn
df

,top_n,gnnsom_topn,smartcyp_topn,xenosite_topn,somp_topn
0,top1_recall,0.138211,0.105691,0.268293,0.154472
1,top3_recall,0.430894,0.300813,0.504065,0.357724
2,top5_recall,0.609756,0.455285,0.634146,0.430894
